In [1]:
import sys
sys.path.append('..') # Allows notebook to find 'src'

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import requests

from src.event_core import process_event_request_simple, load_state

In [2]:
def get_available_models():
    # Get local models from Ollama
    try:
        response = requests.get('http://localhost:11434/api/tags')
        response.raise_for_status()
        models = response.json().get('models', [])
        ollama_models = [model['name'] for model in models] if models else []
    except requests.exceptions.RequestException:
        ollama_models = []
    
    # Add a curated list of high-performance Together AI models
    together_models = [
        'togetherai/meta-llama/Llama-3-8b-chat-hf', # Good & fast
        'togetherai/mistralai/Mixtral-8x7B-Instruct-v0.1', # Very powerful
        'togetherai/Qwen/Qwen1.5-7B-Chat' # Another strong option
    ]
    
    return ollama_models + together_models

# --- UI WIDGETS ---
header = widgets.HTML("<h1>Event Management Assistant 🤖</h1><p>Welcome! I can help you manage your conference venues and sessions.</p>")
available_models = get_available_models()
model_selector = widgets.Dropdown(options=available_models, value=(available_models[0] if available_models else None), description='Select Model:', style={'description_width': 'initial'})
role_selector = widgets.RadioButtons(options=['admin', 'scheduler'], value='admin', description='Your Role:')
state_html_view = widgets.HTML()
state_accordion = widgets.Accordion(children=[state_html_view], titles=('Current Conference State',))
chat_history_box = widgets.VBox([])
user_input = widgets.Text(placeholder='Type your request here...', layout={'width': '95%'})
send_button = widgets.Button(description="Send", button_style='success', icon='paper-plane')

# --- LOGIC TO UPDATE STATE DISPLAY ---
def update_state_display():
    try:
        current_state = load_state()
        venues = current_state.get('venues', {})
        sessions = current_state.get('sessions', [])
        bookings = current_state.get('venue_bookings', {})
        venues_html = "<h4>Venues</h4><table border='1' style='width:100%; border-collapse: collapse;'><tr><th style='padding:5px;text-align:left;'>Name</th><th style='padding:5px;text-align:left;'>Capacity</th><th style='padding:5px;text-align:left;'>Has A/V</th><th style='padding:5px;text-align:left;'>Status</th></tr>"
        if not venues: venues_html += "<tr><td colspan='4' style='padding:5px;'><i>No venues created yet.</i></td></tr>"
        else:
            for name, props in sorted(venues.items()):
                status = "Booked" if name in bookings else "Available"
                status_color = "#FADBD8" if status == "Booked" else "#D5F5E3"
                venues_html += f"<tr><td style='padding:5px;'>{name}</td><td style='padding:5px;'>{int(props.get('capacity', 0))}</td><td style='padding:5px;'>{props.get('has_av_system', 'N/A')}</td><td style='padding:5px;background-color:{status_color};'>{status}</td></tr>"
        venues_html += "</table>"
        sessions_html = "<h4 style='margin-top: 15px;'>Scheduled Sessions</h4><table border='1' style='width:100%; border-collapse: collapse;'><tr><th style='padding:5px;text-align:left;'>Name</th><th style='padding:5px;text-align:left;'>Venue</th><th style='padding:5px;text-align:left;'>Host</th><th style='padding:5px;text-align:left;'>Attendees</th></tr>"
        if not sessions: sessions_html += "<tr><td colspan='4' style='padding:5px;'><i>No sessions scheduled yet.</i></td></tr>"
        else:
            for session in sessions:
                sessions_html += f"<tr><td style='padding:5px;'>{session.get('name', 'N/A')}</td><td style='padding:5px;'>{session.get('in_venue', 'N/A')}</td><td style='padding:5px;'>{session.get('hosted_by', 'N/A')}</td><td style='padding:5px;'>{session.get('expected_attendees', 'N/A')}</td></tr>"
        sessions_html += "</table>"
        state_html_view.value = venues_html + sessions_html
    except Exception as e:
        state_html_view.value = f"<p style='color:red;'>Error loading state: {e}</p>"

# --- CHAT LOGIC WITH HISTORY ---
conversation_history = []

def on_send_button_clicked(b):
    query = user_input.value
    if not query: return
    send_button.disabled = True
    user_input.value = ""
    
    conversation_history.append({'role': 'user', 'content': query})
    user_msg = widgets.HTML(f"<div class='chat-bubble user-bubble'><b>User:</b> {query}</div>")
    assistant_placeholder = widgets.HTML("<div class='chat-bubble assistant-bubble'><i>Assistant is thinking...</i></div>")
    chat_history_box.children = list(chat_history_box.children) + [user_msg, assistant_placeholder]
    
    result = process_event_request_simple(conversation_history, role_selector.value, model_selector.value)
    
    conversation_history.append({'role': 'assistant', 'content': result['message']})
    final_msg = widgets.HTML(f"<div class='chat-bubble assistant-bubble'><b>Assistant:</b><br>{result['message']}</div>")
    chat_history_box.children = list(chat_history_box.children)[:-1] + [final_msg]

    if result['status'] == 'success':
        update_state_display()

    send_button.disabled = False

# --- INITIAL UI DISPLAY ---
send_button.on_click(on_send_button_clicked)
display(HTML("""<style>.chat-bubble{max-width:80%;padding:10px;border-radius:10px;margin-top:5px;margin-bottom:5px;}.user-bubble{background-color:#EBF5FB;align-self:flex-end;margin-left:20%;}.assistant-bubble{background-color:#E8F8F5;align-self:flex-start;margin-right:20%;}</style>"""))
update_state_display()
initial_msg_text = "Hello! I'm ready to help you plan your event."
conversation_history.append({'role': 'assistant', 'content': initial_msg_text})
initial_msg_widget = widgets.HTML(f"<div class='chat-bubble assistant-bubble'><b>Assistant:</b><br>{initial_msg_text}</div>")
chat_history_box.children = [initial_msg_widget]

display(widgets.VBox([
    header, widgets.HBox([model_selector, role_selector]), widgets.HTML("<hr>"),
    state_accordion, chat_history_box, widgets.HBox([user_input, send_button]),
]))